# BugBuster Desktop App

Builds and launches the **Tauri + Leptos** desktop application.

| Cell | Action |
|------|--------|
| Setup | Verify toolchain and paths |
| Dev mode | `cargo tauri dev` — hot-reload, opens window automatically |
| Release build | `cargo tauri build` — produces installer/exe |
| Launch | Run the already-built executable directly |

In [ ]:
import subprocess
import os
import sys
import threading
from pathlib import Path
from IPython.display import display, HTML

APP_DIR = Path(r"C:\Users\Public\Documents\Altium\BugBuster\DesktopApp\BugBuster")

def check_tool(cmd, args=["--version"]):
    try:
        r = subprocess.run([cmd] + args, capture_output=True, text=True)
        return r.stdout.strip() or r.stderr.strip()
    except FileNotFoundError:
        return None

tools = {
    "rustc":      check_tool("rustc"),
    "cargo":      check_tool("cargo"),
    "trunk":      check_tool("trunk"),
    "cargo-tauri": check_tool("cargo", ["tauri", "--version"]),
}

print("=== Toolchain check ===")
all_ok = True
for name, ver in tools.items():
    status = "OK" if ver else "MISSING"
    print(f"  {name:<15} {status}  {ver or ''}")
    if not ver:
        all_ok = False

print()
print(f"App directory exists: {APP_DIR.exists()}")
print(f"Path: {APP_DIR}")
if not all_ok:
    print("\nWARNING: Some tools are missing. Install them before running.")

In [ ]:
# ─── Stream subprocess output live ───────────────────────────────────────────
import signal

_active_proc = None

def run_streaming(cmd, cwd=APP_DIR, env=None):
    """Run a command and stream stdout/stderr to the notebook in real-time."""
    global _active_proc
    merged_env = {**os.environ, **(env or {})}
    print(f"$ {' '.join(cmd)}")
    print("-" * 60)
    proc = subprocess.Popen(
        cmd,
        cwd=str(cwd),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=merged_env,
    )
    _active_proc = proc
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
    except KeyboardInterrupt:
        proc.terminate()
        print("\n[Interrupted — process terminated]")
    finally:
        proc.wait()
        _active_proc = None
    print("-" * 60)
    print(f"Exited with code {proc.returncode}")
    return proc.returncode

def stop_app():
    """Terminate the running app process (call from a new cell if needed)."""
    global _active_proc
    if _active_proc and _active_proc.poll() is None:
        _active_proc.terminate()
        print("Process terminated.")
    else:
        print("No active process.")

print("Helpers loaded. Use stop_app() in a new cell to terminate a running process.")

## Development mode
Starts `trunk serve` (frontend) + Tauri backend simultaneously.  
The app window opens automatically. **This cell runs until you stop it** (Kernel → Interrupt or call `stop_app()`).

In [ ]:
run_streaming(["cargo", "tauri", "dev"])

## Release build
Compiles an optimised binary and bundles an installer under `src-tauri/target/release/bundle/`.

In [ ]:
run_streaming(["cargo", "tauri", "build"])

## Launch already-built executable
Finds and launches the `.exe` produced by the release build without rebuilding.

In [ ]:
import glob

# Search for the built executable
search_patterns = [
    str(APP_DIR / "src-tauri" / "target" / "release" / "*.exe"),
    str(APP_DIR / "src-tauri" / "target" / "release" / "bundle" / "nsis" / "*.exe"),
    str(APP_DIR / "src-tauri" / "target" / "release" / "bundle" / "msi" / "*.msi"),
]

candidates = []
for pattern in search_patterns:
    candidates.extend(glob.glob(pattern))

# Filter out setup installers — prefer the raw .exe
exes = [p for p in candidates if "bugbuster" in Path(p).name.lower() and not p.endswith(".msi")]

if not exes:
    print("No built executable found. Run the 'Release build' cell first.")
    print("Searched:", search_patterns)
else:
    exe = exes[0]
    print(f"Launching: {exe}")
    subprocess.Popen([exe], cwd=str(APP_DIR))
    print("App started (detached — window will open shortly).")

## Stop the running process
Run this cell to terminate `cargo tauri dev` if still running.

In [ ]:
stop_app()